# Skin Lesion Classification with DINOv2

Train a DINOv2 vision foundation model on the ISIC 2019 skin lesion dataset.

**Runtime:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook clones the repo and calls into the project scripts — all core logic lives in `src/`.

### Workflow
1. Download data
2. Extract DINOv2 features once (~5 min on T4)
3. Train linear head on cached features (~30 sec)
4. Evaluate

## 1. Setup

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# Clone the repo (only needs to run once per session)
import os

REPO_DIR = "/content/isic-skin-lesion-classification"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mtang4/isic-skin-lesion-classification.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

## 2. Download and Prepare Data

Downloads the ISIC 2019 dataset (~9GB) and creates stratified train/val/test splits.

This takes ~10-15 minutes on Colab. The data is stored in `/content/isic-skin-lesion-classification/data/` and persists for the session.

In [ ]:
!python scripts/download_data.py --data_dir ./data

In [ ]:
# Quick sanity check: view class distribution
import pandas as pd

weights = pd.read_csv("data/class_weights.csv")
print(weights.to_string(index=False))
print(f"\nTotal training samples: {weights['count'].sum()}")

In [ ]:
# Preview some training images
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random

CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, cls in zip(axes.flat, CLASS_NAMES):
    cls_dir = Path(f"data/train/{cls}")
    img_path = random.choice(list(cls_dir.glob("*.jpg")))
    ax.imshow(Image.open(img_path))
    ax.set_title(cls, fontsize=14)
    ax.axis("off")
plt.suptitle("Sample Training Images (one per class)", fontsize=16)
plt.tight_layout()
plt.show()

## 3. Extract Features

Run every image through the frozen DINOv2 backbone **once** and cache the 768-dim
`[CLS]` embeddings to disk. This takes ~5 minutes on a T4 and means we never need
to run the backbone again during training.

In [ ]:
BACKBONE = "facebook/dinov2-base"
FEATURE_DIR = f"data/features/{BACKBONE.replace('/', '_')}"

!python src/extract_features.py --data_dir ./data --backbone {BACKBONE}

## 4. Train Linear Head

Train a linear classifier on the cached features. This is essentially logistic
regression on 768-dim vectors — takes ~30 seconds for 30 epochs.

In [ ]:
!python src/train_cached.py \
    --feature_dir {FEATURE_DIR} \
    --epochs 30 \
    --lr 1e-3 \
    --run_name cached_linear_base

## 5. Evaluate

In [ ]:
CHECKPOINT = "checkpoints/cached_linear_base/best_head.pt"

!python src/evaluate_cached.py \
    --feature_dir {FEATURE_DIR} \
    --checkpoint {CHECKPOINT}

In [ ]:
# Display the generated plots
from IPython.display import display, Image as IPImage
from pathlib import Path

ckpt_dir = Path(CHECKPOINT).parent

for plot in ["confusion_matrix.png", "per_class_metrics.png"]:
    plot_path = ckpt_dir / plot
    if plot_path.exists():
        print(f"\n{plot}")
        display(IPImage(filename=str(plot_path), width=800))

## 6. (Optional) Full Image Training

The scripts above are the fast path. If you want to fine-tune the full backbone
(not just the linear head), use `train.py` which processes raw images.
This is slower (~1hr on T4) but can yield better results.

In [ ]:
# Uncomment to run full fine-tuning
# !python src/train.py \
#     --config configs/default.yaml \
#     --mode fine_tune \
#     --epochs 15 \
#     --lr 1e-4 \
#     --batch_size 32 \
#     --run_name finetune_base

## 7. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/

## 8. Save Checkpoint to Google Drive (optional)

Colab sessions are ephemeral. Save your best checkpoint and features to Drive so you don't lose them.

In [ ]:
# Uncomment to mount Drive and copy artifacts
# from google.colab import drive
# import shutil
# drive.mount("/content/drive")
#
# drive_dest = "/content/drive/MyDrive/ml-checkpoints/isic-dinov2/"
# os.makedirs(drive_dest, exist_ok=True)
# shutil.copy2(CHECKPOINT, drive_dest)
#
# # Also save cached features so you don't need the GPU next time
# drive_feat = "/content/drive/MyDrive/ml-checkpoints/isic-dinov2/features/"
# shutil.copytree(FEATURE_DIR, drive_feat, dirs_exist_ok=True)
# print(f"Saved to {drive_dest}")